# 🐍 Live Coding: Декоратори в Python
### Урок побудований на проблемі — від болю до рішення

---

> **Нотатка для викладача:** Не розкривай тему декораторів одразу. Дай студентам відчути проблему самостійно. Кожен етап — це крок до розуміння, а не лекція.

---
## 🎬 Вступ

Уявіть, що ми пишемо простий блог. У блозі є дії: переглядати пости, редагувати, видаляти, публікувати.

І є **три ролі** користувачів:
- `guest` — гість, може тільки читати
- `user` — звичайний користувач
- `admin` — адміністратор, може все

Наше завдання — зробити так, щоб кожна дія перевіряла, чи є у користувача право її виконувати.

**Починаємо!** 🚀

---
## 📦 Етап 1 — Наївна реалізація (без декораторів)

Просто напишемо код "в лоб". Перший інстинкт будь-якого програміста — використати `if`.

In [8]:
# Поточний користувач — можна змінювати для тестування
current_user = {}

In [9]:
def view_post(post_id):
    # Гості теж можуть читати — дозволяємо всім
    if current_user["role"] not in ["guest", "user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


def create_post(title):
    # Гостям не можна створювати пости
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


def edit_post(post_id):
    # Тільки user і admin
    if current_user["role"] not in ["user", "admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post(post_id):
    # Публікувати може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


def delete_post(post_id):
    # Видаляти може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


def archive_post(post_id):
    # Архівувати може тільки admin
    if current_user["role"] not in ["admin"]:
        print("❌ Доступ заборонено!")
        return
    print(f"📁  {current_user['name']} архівує пост #{post_id}")

In [10]:
# Тестуємо як гість
current_user = {"name": "Іван", "role": "guest"}

view_post(1)
create_post("Мій перший пост")
delete_post(1)

👁️  Іван переглядає пост #1
❌ Доступ заборонено!
❌ Доступ заборонено!


In [11]:
# Тестуємо як admin
current_user = {"name": "Оля", "role": "admin"}

view_post(1)
create_post("Важливе оголошення")
delete_post(1)

👁️  Оля переглядає пост #1
✍️  Оля створює пост: 'Важливе оголошення'
🗑️  Оля видаляє пост #1


---
## 😬 Етап 2 — Показуємо біль

Окей, код працює. Але... подивіться на нього уважно.

**Ось тут починається біль** 👇

Порахуйте, скільки разів у нас зустрічається ось цей шматок:

```python
if current_user["role"] not in [...]:
    print("❌ Доступ заборонено!")
    return
```

**6 разів!** І це тільки у 6 функціях.

А тепер уявіть:

❓ **Що буде, якщо у нас не 6 функцій, а 60?**

❓ **Що буде, якщо треба змінити повідомлення про помилку з** `"❌ Доступ заборонено!"` **на** `"🚫 У вас немає прав!"`?

❓ **Що буде, якщо треба додати логування — записувати кожну невдалу спробу доступу?**

**Відповідь:** Доведеться зайти в кожну функцію і вручну щось змінювати. 6 разів. 60 разів. 600 разів.

Це і є дублювання коду. Логіка доступу **розкидана** по всіх функціях. Вона не в одному місці — вона скрізь.

І це проблема.

---
## 🔥 Етап 3 — Погіршуємо ситуацію

Хочете більше болю? 😅

До нас приходить менеджер і каже: **"Додайте нову роль — `moderator`. Він може редагувати і публікувати, але не може видаляти."**

Дивіться, що нам доведеться робити...

In [6]:
# Нам ВРУЧНУ треба зайти в кожну функцію і додати moderator

def view_post_v2(post_id):
    if current_user["role"] not in ["guest", "user", "admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


def edit_post_v2(post_id):
    if current_user["role"] not in ["user", "admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


def publish_post_v2(post_id):
    if current_user["role"] not in ["admin", "moderator"]:  # ← додали
        print("❌ Доступ заборонено!")
        return
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


def delete_post_v2(post_id):
    if current_user["role"] not in ["admin"]:  # ← moderator НЕ може видаляти!
        print("❌ Доступ заборонено!")
        return
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")

# ... і так далі ще в 2 функціях

print("Ми тільки що вручну зайшли в 4+ функції заради ONE нової ролі 🤦")

Ми тільки що вручну зайшли в 4+ функції заради ONE нової ролі 🤦


Ось картина після зміни:

| Функція | Хто має доступ |
|---------|----------------|
| `view_post` | guest, user, admin, **moderator** |
| `create_post` | user, admin |
| `edit_post` | user, admin, **moderator** |
| `publish_post` | admin, **moderator** |
| `delete_post` | admin |
| `archive_post` | admin |

І щоразу, коли бізнес-правила змінюються — ми знову руками лізем в кожну функцію.

**Це не масштабується. Це крихко. Це проблема.**

---
## 💡 Етап 4 — Ідея рішення

Давайте подумаємо разом.

Що якби ми могли **"загорнути"** будь-яку функцію в перевірку доступу?

Ось псевдокод ідеї:

```
ПЕРЕД виконанням функції → перевір роль
  якщо роль підходить → виконай функцію
  якщо ні → заблокуй
```

Тобто нам потрібна якась **"обгортка"** — функція, яка приймає іншу функцію і додає до неї перевірку доступу.

Спробуємо написати це самостійно, крок за кроком.

In [7]:
# Крок 1: Функції є об'єктами в Python
# Їх можна передавати як аргументи!

def say_hello():
    print("Привіт!")


def run_function(func):
    print("--- Запускаємо функцію ---")
    func()  # викликаємо функцію, яку передали
    print("--- Готово ---")


run_function(say_hello)  # передаємо say_hello БЕЗ дужок!

--- Запускаємо функцію ---
Привіт!
--- Готово ---


In [10]:
# Крок 2: Функція може ПОВЕРТАТИ іншу функцію
# Це трохи незвично, але дуже потужно

def create_greeter(name):
    # Всередині визначаємо нову функцію
    def greet():
        print(f"Привіт, {name}!")
    # І повертаємо її (без виклику!)
    return greet


greet_ivan = create_greeter("Іван")
greet_olia = create_greeter("Оля")

greet_ivan()
greet_olia()

Привіт, Іван!
Привіт, Оля!


In [11]:
# Крок 3: Поєднуємо обидві ідеї
# Приймаємо функцію → повертаємо нову функцію з перевіркою

def require_admin(func):
    """Обгортка: перевіряє роль перед виконанням"""

    def wrapper():
        # Ось вся логіка перевірки — в одному місці!
        if current_user["role"] != "admin":
            print("❌ Доступ заборонено! Потрібна роль: admin")
            return
        # Якщо все ок — виконуємо оригінальну функцію
        func()

    return wrapper


# Тестуємо
def delete_everything():
    print("🗑️ Видалено все!")


# «Загортаємо» функцію в захист
safe_delete = require_admin(delete_everything)

current_user = {"name": "Гість", "role": "guest"}
safe_delete()  # заблоковано

current_user = {"name": "Адмін", "role": "admin"}
safe_delete()  # дозволено

❌ Доступ заборонено! Потрібна роль: admin
🗑️ Видалено все!


Дивіться — логіка доступу тепер **в одному місці**!

Саме це і є ідея декоратора. А Python просто дає нам **зручний синтаксис** для цього.

---
## 🎯 Етап 5 — Знайомство з декоратором

Те, що ми зробили вище — це і є декоратор. Але Python дозволяє записати це красиво через `@`.

Замість:
```python
safe_delete = require_admin(delete_everything)
```

Можна писати:
```python
@require_admin
def delete_everything():
    ...
```

Це **рівно одне й те саме**. `@` — це просто скорочення. Синтаксичний цукор.

Тепер зробимо універсальний декоратор — не для конкретної ролі, а для **будь-якої**.

In [1]:
import functools


def require_role(*allowed_roles):
    """
    Декоратор-фабрика: приймає список дозволених ролей
    і повертає декоратор, який перевіряє роль поточного користувача.
    """
    def decorator(func):
        @functools.wraps(func)  # зберігаємо ім'я та документацію оригінальної функції
        def wrapper(*args, **kwargs):
            # Перевіряємо роль — ВСЯ логіка тут, в одному місці
            if current_user["role"] not in allowed_roles:
                print(f"❌ Доступ заборонено! Потрібна роль: {' або '.join(allowed_roles)}")
                return
            # Роль підходить — викликаємо оригінальну функцію
            # *args, **kwargs — передаємо ВСІ аргументи, які прийшли у wrapper
            return func(*args, **kwargs)

        return wrapper
    return decorator

**Розберемо по частинах:**

| Що | Навіщо |
|----|--------|
| `require_role(*allowed_roles)` | Зовнішня функція — приймає список ролей (`"admin"`, `"user"`, ...) |
| `decorator(func)` | Середня функція — приймає функцію, яку «захищаємо» |
| `wrapper(*args, **kwargs)` | Внутрішня функція — те, що реально виконується замість оригіналу |
| `*args, **kwargs` | «Передай усе далі» — щоб декоратор підходив для будь-якої функції |
| `@functools.wraps(func)` | Щоб `delete_post.__name__` лишався `"delete_post"`, а не `"wrapper"` |

---
## ✨ Етап 6 — Рефакторинг: переписуємо все через декоратор

Тепер подивіться, яким чистим стає код!

In [2]:
# ✅ ПІСЛЯ РЕФАКТОРИНГУ — логіка доступу прибрана з тіла функцій

@require_role("guest", "user", "admin", "moderator")
def view_post(post_id):
    print(f"👁️  {current_user['name']} переглядає пост #{post_id}")


@require_role("user", "admin")
def create_post(title):
    print(f"✍️  {current_user['name']} створює пост: '{title}'")


@require_role("user", "admin", "moderator")
def edit_post(post_id):
    print(f"✏️  {current_user['name']} редагує пост #{post_id}")


@require_role("admin", "moderator")
def publish_post(post_id):
    print(f"📢  {current_user['name']} публікує пост #{post_id}")


@require_role("admin")
def delete_post(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


@require_role("admin")
def archive_post(post_id):
    print(f"📁  {current_user['name']} архівує пост #{post_id}")


print("Декоратори задекларовано ✅")

Декоратори задекларовано ✅


In [3]:
# Тестуємо — гість
current_user = {"name": "Гість Анонім", "role": "guest"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
create_post("Мій пост")
delete_post(42)

=== Користувач: Гість Анонім (роль: guest) ===
👁️  Гість Анонім переглядає пост #42
❌ Доступ заборонено! Потрібна роль: user або admin
❌ Доступ заборонено! Потрібна роль: admin


In [4]:
# Тестуємо — модератор
current_user = {"name": "Марко-Модератор", "role": "moderator"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
edit_post(42)
publish_post(42)
delete_post(42)   # ← не зможе

=== Користувач: Марко-Модератор (роль: moderator) ===
👁️  Марко-Модератор переглядає пост #42
✏️  Марко-Модератор редагує пост #42
📢  Марко-Модератор публікує пост #42
❌ Доступ заборонено! Потрібна роль: admin


In [5]:
# Тестуємо — адмін
current_user = {"name": "Адмін Всесильний", "role": "admin"}
print(f"=== Користувач: {current_user['name']} (роль: {current_user['role']}) ===")

view_post(42)
create_post("Важливий анонс")
edit_post(42)
publish_post(42)
delete_post(42)
archive_post(42)

=== Користувач: Адмін Всесильний (роль: admin) ===
👁️  Адмін Всесильний переглядає пост #42
✍️  Адмін Всесильний створює пост: 'Важливий анонс'
✏️  Адмін Всесильний редагує пост #42
📢  Адмін Всесильний публікує пост #42
🗑️  Адмін Всесильний видаляє пост #42
📁  Адмін Всесильний архівує пост #42


### Порівняємо «до» і «після»

**До рефакторингу** — функція `delete_post`:
```python
def delete_post(post_id):
    if current_user["role"] not in ["admin"]:   # ← ця логіка
        print("❌ Доступ заборонено!")            # ← повторюється
        return                                   # ← 6 разів
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")
```

**Після рефакторингу** — функція `delete_post`:
```python
@require_role("admin")             # ← одна строчка, все зрозуміло
def delete_post(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")
```

Тіло функції тепер містить **тільки корисну логіку**. Більше нічого зайвого.

---
## 🎁 Бонус: додаємо нову роль — одна зміна!

Пам'ятаєте, яким болем було додавати `moderator` раніше? Тепер додамо роль `superuser` — і жодну функцію чіпати не будемо.

In [6]:
# Просто оновлюємо декоратори — і все

@require_role("admin", "superuser")  # ← додали superuser тут
def delete_post_v3(post_id):
    print(f"🗑️  {current_user['name']} видаляє пост #{post_id}")


@require_role("admin", "superuser")  # ← і тут
def archive_post_v3(post_id):
    print(f"📁  {current_user['name']} архівує пост #{post_id}")


# Тестуємо superuser
current_user = {"name": "Супер Юзер", "role": "superuser"}
delete_post_v3(99)
archive_post_v3(99)

🗑️  Супер Юзер видаляє пост #99
📁  Супер Юзер архівує пост #99


---
## 🏁 Етап 7 — Висновок

### Що таке декоратор?

Декоратор — це функція, яка **загортає** іншу функцію і додає їй нову поведінку.

Ніякої магії немає. Це просто:
1. Функція, яка приймає функцію
2. Всередині робить щось додатково
3. Повертає нову функцію

```
🔧 require_role("admin")
        ↓
   загортає
        ↓
📦 delete_post
        ↓
  у нову функцію, яка
  спочатку перевіряє роль,
  потім (якщо ок) викликає оригінал
```

### Коли використовувати декоратори?

Якщо помічаєте, що **один і той самий шматок коду** повторюється на початку або в кінці багатьох функцій — це сигнал. Зупиніться і подумайте: "Чи не час тут декоратор?"

Класичні приклади:
- ✅ Перевірка прав доступу (як у нас)
- ✅ Логування (записати в лог, що функція була викликана)
- ✅ Кешування (не рахувати те, що вже рахували)
- ✅ Вимірювання часу виконання
- ✅ Повторна спроба при помилці (retry)

### Головна ідея

> **Декоратор відповідає на питання «ЯК запустити функцію», а не «ЩО вона робить».**

Функція `delete_post` знає тільки одне: як видалити пост. Вона не повинна знати про ролі, логування, кешування. Це не її робота. Для цього є декоратори.

In [7]:
# 🎓 Фінальний «живий" приклад для закріплення
# Напишемо декоратор-таймер — щоб зрозуміти, що декоратори — це загальна ідея

import time
import functools


def timer(func):
    """Вимірює час виконання функції"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()           # запам'ятовуємо час до
        result = func(*args, **kwargs) # виконуємо оригінальну функцію
        end = time.time()             # запам'ятовуємо час після
        print(f"⏱️  {func.__name__} виконалась за {end - start:.4f} сек")
        return result
    return wrapper


@timer
def heavy_calculation(n):
    """Симулюємо важкі обчислення"""
    total = sum(range(n))
    return total


@timer
def fast_calculation(n):
    """Швидка формула Гауса"""
    return n * (n - 1) // 2


print(heavy_calculation(1_000_000))
print(fast_calculation(1_000_000))

⏱️  heavy_calculation виконалась за 0.0170 сек
499999500000
⏱️  fast_calculation виконалась за 0.0000 сек
499999500000


---

## 📚 Підсумкова шпаргалка

```python
import functools

# Декоратор без параметрів
def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # робимо щось ДО
        result = func(*args, **kwargs)
        # робимо щось ПІСЛЯ
        return result
    return wrapper


# Декоратор З параметрами (фабрика)
def my_decorator_with_args(param):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # використовуємо param тут
            return func(*args, **kwargs)
        return wrapper
    return decorator


# Використання
@my_decorator
def foo():
    pass

@my_decorator_with_args("значення")
def bar():
    pass
```

---

**Молодці! 🎉** Тепер ви знаєте, що таке декоратори — і головне, *навіщо* вони потрібні.